In [1]:
!pip install -q ultralytics easyocr filterpy lap scipy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 29.1 MB/s eta 0:00:00


In [2]:
import cv2
import re
import os
import math
import time
import easyocr
import numpy as np
import pandas as pd

from ultralytics import YOLO
from collections import Counter, defaultdict
from google.colab.patches import cv2_imshow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# ==========================================
# PATHS FOR THE CAR VIDEO
# ==========================================
import os

MODEL_PATH = "/content/best.pt"
OUTPUT_VIDEO = "/content/output/VIDEO.mp4"
OUTPUT_CSV = "/content/outputVIDEO.csv"

# Make sure output directory exists dynamically
os.makedirs("/content/output", exist_ok=True)

# ─── TARGETING THE ORIGINAL SINGLE INPUT VIDEO STREAM ───
VIDEO_PATH = "/content/EXP VIDEO.mp4"

print(f" Input Video Path Reset to Original: {VIDEO_PATH}")

# ==========================================
# STABLE CONFIGURATIONS FOR THE ORIGINAL TRACK
# ==========================================
YOLO_CONF = 0.35
OCR_INTERVAL = 1
LOCK_THRESHOLD = 4
OCR_CONFIDENCE = 0.20

MAX_HISTORY = 20
MIN_PLATE_WIDTH = 45
MIN_PLATE_HEIGHT = 15
UPSCALE_FACTOR = 3

 Input Video Path Reset to Original: /content/EXP VIDEO.mp4


In [4]:
print("Loading YOLO...")
model = YOLO(MODEL_PATH)

print("Loading EasyOCR...")
reader = easyocr.Reader(['en'], gpu=True)

print("Everything Loaded Successfully ")

Loading YOLO...
Loading EasyOCR...


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEverything Loaded Successfully 


In [5]:
def preprocess_plate(plate):
    processed_images = []

    # Dynamic Upscaling
    plate = cv2.resize(
        plate,
        None,
        fx=UPSCALE_FACTOR,
        fy=UPSCALE_FACTOR,
        interpolation=cv2.INTER_CUBIC
    )
    processed_images.append(plate)

    # Grayscale Layer Transformation
    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    processed_images.append(gray)

    # Contrast Adjustment Layer
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray)
    processed_images.append(clahe_img)

    # Gaussian Blur Layer
    blur = cv2.GaussianBlur(clahe_img, (3,3), 0)
    processed_images.append(blur)

    # Adaptive Threshold Filtering
    adaptive = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 2
    )
    processed_images.append(adaptive)

    # Otsu Binarization Implementation
    _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    processed_images.append(otsu)

    # Sharpening Filter Kernel Matrix
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharp = cv2.filter2D(blur, -1, kernel)
    processed_images.append(sharp)

    return processed_images

In [6]:
def read_plate(plate_img):
    if plate_img.size == 0:
        return "", 0.0

    gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (0, 0), fx=UPSCALE_FACTOR, fy=UPSCALE_FACTOR, interpolation=cv2.INTER_LINEAR)

    # Alphanumeric parsing with explicit allowed dataset elements
    results = reader.readtext(
        gray,
        paragraph=False,
        allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-'
    )

    if not results:
        return "", 0.0

    text = results[0][1]
    score = results[0][2]

    return text, score

In [7]:
import re
import math
import numpy as np
from collections import Counter, defaultdict

class IndianPlateMemoryManager:
    def __init__(self, max_history=25, lock_threshold=4):
        self.vehicle_memory = defaultdict(list)
        self.vehicle_plate = {}
        self.locked_plates = {}
        self.last_known_positions = {}
        self.id_mapping_bridge = {}
        self.max_history = max_history
        self.lock_threshold = lock_threshold  # Increased for stricter dynamic locking

        # Phase 2 Timing Structures Aligned
        self.arrival_frames = {}
        self.departure_frames = {}
        self.waiting_frames_count = defaultdict(int)

    def clean_plate(self, text):
        if not text:
            return ""
        text = text.upper()
        text = re.sub(r'[^A-Z0-9]', '', text)
        return text

    def frame_to_clock_time(self, frame_count, fps, base_hour=3, base_min=0, base_sec=0):
        elapsed_seconds = frame_count / fps
        total_seconds = base_hour * 3600 + base_min * 60 + base_sec + elapsed_seconds
        hours = int((total_seconds // 3600) % 24)
        minutes = int((total_seconds % 3600) // 60)
        seconds = int(total_seconds % 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    def duration_to_clock_format(self, stopped_frames, fps):
        total_seconds = stopped_frames / fps
        hours = int(total_seconds // 3600)
        minutes = int((total_seconds % 3600) // 60)
        seconds = int(total_seconds % 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    def dynamic_spatial_id_recovery(self, track_id, box):
        x1, y1, x2, y2 = box
        current_center = ((x1 + x2) / 2, (y1 + y2) / 2)

        resolved_id = track_id
        while resolved_id in self.id_mapping_bridge:
            resolved_id = self.id_mapping_bridge[resolved_id]

        closest_id = None
        min_distance = 60.0

        for hist_id, hist_center in self.last_known_positions.items():
            if hist_id == resolved_id:
                continue
            dist = math.sqrt((current_center[0] - hist_center[0])**2 + (current_center[1] - hist_center[1])**2)
            if dist < min_distance:
                min_distance = dist
                closest_id = hist_id

        if closest_id is not None:
            base_resolved_id = closest_id
            while base_resolved_id in self.id_mapping_bridge:
                base_resolved_id = self.id_mapping_bridge[base_resolved_id]
            self.id_mapping_bridge[track_id] = base_resolved_id
            resolved_id = base_resolved_id

        return resolved_id, current_center

    def stabilize_text_algorithmically(self, history):
        if not history:
            return ""
        lengths = [len(text) for text, conf in history]
        dominant_length = Counter(lengths).most_common(1)[0][0]
        filtered_records = [item for item in history if len(item[0]) == dominant_length]

        final_chars = []
        for idx in range(dominant_length):
            char_weights = {}
            for text, conf in filtered_records:
                if idx < len(text):
                    char = text[idx]
                    char_weights[char] = char_weights.get(char, 0.0) + conf
            best_char = max(char_weights, key=char_weights.get)
            final_chars.append(best_char)

        return "".join(final_chars)

    def update_analytics(self, track_id, box, frame_count):
        resolved_id, current_center = self.dynamic_spatial_id_recovery(track_id, box)

        if resolved_id not in self.arrival_frames:
            self.arrival_frames[resolved_id] = frame_count

        if resolved_id in self.last_known_positions:
            prev_center = self.last_known_positions[resolved_id]
            motion_delta = math.sqrt((current_center[0] - prev_center[0])**2 + (current_center[1] - prev_center[1])**2)
            if motion_delta < 1.5:
                self.waiting_frames_count[resolved_id] += 1

        self.departure_frames[resolved_id] = frame_count
        self.last_known_positions[resolved_id] = current_center
        return resolved_id

    def update_plate(self, track_id, box, text, confidence, frame_count):
        resolved_id = self.update_analytics(track_id, box, frame_count)

        if resolved_id in self.locked_plates:
            return resolved_id

        text = self.clean_plate(text)
        if len(text) < 5:
            return resolved_id

        # Drop shorter fuzzy reads if a strong baseline is already forming
        if resolved_id in self.vehicle_plate:
            existing_len = len(self.vehicle_plate[resolved_id])
            if len(text) < existing_len and existing_len >= 9:
                return resolved_id

        self.vehicle_memory[resolved_id].append((text, confidence))
        if len(self.vehicle_memory[resolved_id]) > self.max_history:
            self.vehicle_memory[resolved_id].pop(0)

        best_plate = self.stabilize_text_algorithmically(self.vehicle_memory[resolved_id])

        # ─── PURE DYNAMIC ALGORITHMIC LOCK (NO HARDCODING) ───
        if len(best_plate) >= 9:
            self.vehicle_plate[resolved_id] = best_plate

            raw_strings = [item[0] for item in self.vehicle_memory[resolved_id]]
            if raw_strings.count(best_plate) >= self.lock_threshold:
                self.locked_plates[resolved_id] = best_plate

        return resolved_id

    def get_plate(self, resolved_id):
        if resolved_id in self.locked_plates:
            return self.locked_plates[resolved_id]
        if resolved_id in self.vehicle_plate:
            return self.vehicle_plate[resolved_id]
        return "SCANNING..."

# Re-initialize tracker safely with strict dynamic threshold
tracker_memory = IndianPlateMemoryManager(max_history=25, lock_threshold=4)

In [8]:
import cv2
import pandas as pd
import numpy as np
import os
import subprocess

os.makedirs(os.path.dirname(OUTPUT_VIDEO), exist_ok=True)
TEMP_RAW_VIDEO = "/content/output/temp_raw.mp4"

if os.path.exists(TEMP_RAW_VIDEO): os.remove(TEMP_RAW_VIDEO)
if os.path.exists(OUTPUT_VIDEO): os.remove(OUTPUT_VIDEO)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print(" ERROR: Video file path missing or corrupted.")
else:
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 24

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(TEMP_RAW_VIDEO, fourcc, fps, (frame_width, frame_height))
    frame_count = 0

    print(f" Running Stabilized ANPR with 03:00:00 PM Match Validation Engine...")

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=YOLO_CONF, verbose=False)

        if len(results) == 0 or results[0].boxes is None or results[0].boxes.id is None:
            out.write(frame)
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        scores = results[0].boxes.conf.cpu().numpy()

        for box, track_id, det_conf in zip(boxes, ids, scores):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1

            if w < MIN_PLATE_WIDTH or h < MIN_PLATE_HEIGHT: continue
            plate_crop = frame[max(0, y1):min(frame_height, y2), max(0, x1):min(frame_width, x2)]
            if plate_crop.size == 0: continue

            if frame_count % OCR_INTERVAL == 0:
                text, score = read_plate(plate_crop)
                resolved_id = tracker_memory.update_plate(track_id, (x1, y1, x2, y2), text, score, frame_count)
            else:
                resolved_id = tracker_memory.update_analytics(track_id, (x1, y1, x2, y2), frame_count)

            plate = tracker_memory.get_plate(resolved_id)

            color = (0, 255, 0)
            label = f"ID: {resolved_id} | {plate}"

            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.rectangle(frame, (x1, y1 - text_h - 15), (x1 + text_w + 10, y1), color, -1)
            cv2.putText(frame, label, (x1 + 5, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    try:
        ffmpeg_cmd = f"ffmpeg -y -i '{TEMP_RAW_VIDEO}' -vcodec libx264 -crf 23 -pix_fmt yuv420p '{OUTPUT_VIDEO}' -loglevel error"
        subprocess.run(ffmpeg_cmd, shell=True, check=True)
        if os.path.exists(TEMP_RAW_VIDEO): os.remove(TEMP_RAW_VIDEO)
    except Exception:
        os.rename(TEMP_RAW_VIDEO, OUTPUT_VIDEO)

    # ========================================================
    # PHASE 2: SUMMARY METRICS EXPORT (03:00:00 TIME ALIGNED)
    # ========================================================
    if tracker_memory.arrival_frames:
        summary_data = []
        for tid in tracker_memory.arrival_frames.keys():
            final_plate = tracker_memory.get_plate(tid)
            if final_plate == "SCANNING...": final_plate = "UNKNOWN"

            # Dynamic string formatting to pull matching clock sync matrices
            arr_clock = tracker_memory.frame_to_clock_time(tracker_memory.arrival_frames[tid], fps)
            dep_clock = tracker_memory.frame_to_clock_time(tracker_memory.departure_frames[tid], fps)
            wait_clock = tracker_memory.duration_to_clock_format(tracker_memory.waiting_frames_count[tid], fps)

            summary_data.append({
                "Track_ID": tid,
                "Vehicle_Number": final_plate,
                "Arrival_Time": arr_clock,
                "Departure_Time": dep_clock,
                "Waiting_Time": wait_clock
            })

        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(OUTPUT_CSV, index=False)
        print("\n" + "="*70)
        print(" PARKING METRICS ALIGNED TO VIDEO DIGITAL CLOCK!")
        print(f" Traffic Logs CSV Exported At: {OUTPUT_CSV}")
        print("="*70)

        print("\n--- SYNCHRONIZED PARKING MANAGEMENT DATABASE ---")
        print(summary_df.to_string(index=False))
    else:
        print("No dynamic tracking data logs captured.")

 Running Stabilized ANPR with 03:00:00 PM Match Validation Engine...

 PARKING METRICS ALIGNED TO VIDEO DIGITAL CLOCK!
 Traffic Logs CSV Exported At: /content/outputVIDEO.csv

--- SYNCHRONIZED PARKING MANAGEMENT DATABASE ---
 Track_ID Vehicle_Number Arrival_Time Departure_Time Waiting_Time
        1     RJ19CA9632     03:00:00       03:00:08     00:00:04
